# AL-2002: Artificial Intelligence Lab
## Lab Manual 11 — Single-Layer Perceptron & ANNs using TensorFlow

**Department of Computer Science — FAST-NU, Lahore**  
**Course Instructor:** Dr. Hajra Waheed  
**Lab Instructor:** Ms. Sadia Shafique  
**Section:** BCS-6A | **Semester:** Spring 2026

---

### Lab Tasks Overview
| Task | Title | Type |
|------|-------|------|
| 1 | Customer Churn Prediction | Binary Classification (Tabular) |
| 2 | Medical Image Binary Classification | Binary Classification (Images) |
| 3 | Handwritten Digit Classification | Multi-Class (MNIST) |
| 4 | Flower Image Classification | Multi-Class (Images) |

---
# Task 1: Customer Churn Prediction (Binary Classification)
**Objective:** Predict whether a telecom customer will leave (churn) using an ANN.

**Dataset:** [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
Download `WA_Fn-UseC_-Telco-Customer-Churn.csv` and place it in your working directory.

In [ ]:
# ── Task 1 Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

### Step 1: Load Dataset

In [ ]:
# Load the CSV (update path if needed)
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print("Shape:", df.shape)
df.head()

### Step 2: Preprocessing — Handle Missing Values & Encode Categoricals

In [ ]:
# Drop customerID (not a feature)
df.drop(columns=['customerID'], inplace=True)

# TotalCharges has spaces as missing values — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Handle missing values
print("Missing values before:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
print("\nMissing values after:", df.isnull().sum().sum())

# Encode binary categorical columns
binary_cols = [col for col in df.select_dtypes(include='object').columns if df[col].nunique() == 2]
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# One-hot encode multi-class categoricals
df = pd.get_dummies(df, drop_first=True)

print("\nFinal shape after encoding:", df.shape)
df.head()

In [ ]:
# Separate features and target
X = df.drop(columns=['Churn']).values
y = df['Churn'].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Churn rate in train: {y_train.mean():.2%}")

### Step 3: Build ANN — Input + 2 Hidden Layers (ReLU) + Output (Sigmoid)

In [ ]:
tf.random.set_seed(42)

model_t1 = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid')
], name='ChurnANN')

model_t1.summary()

### Step 4: Compile — Adam + Binary Cross-Entropy

In [ ]:
model_t1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

### Step 5: Train for 20 Epochs

In [ ]:
history_t1 = model_t1.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

### Step 6: Evaluate — Accuracy, Precision, Recall, F1-Score

In [ ]:
# Keras evaluation
results = model_t1.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss:      {results[0]:.4f}")
print(f"Test Accuracy:  {results[1]:.4f}")
print(f"Test Precision: {results[2]:.4f}")
print(f"Test Recall:    {results[3]:.4f}")

# sklearn classification report
y_pred_prob = model_t1.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues')
plt.title('Task 1 — Confusion Matrix: Customer Churn')
plt.tight_layout()
plt.show()

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_t1.history['loss'],     label='Train Loss')
axes[0].plot(history_t1.history['val_loss'], label='Val Loss')
axes[0].set_title('Task 1 — Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_t1.history['accuracy'],     label='Train Accuracy')
axes[1].plot(history_t1.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Task 1 — Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

---
# Task 2: Medical Image Binary Classification (Brain Tumor Detection)
**Objective:** Detect whether a brain MRI shows a tumor or not using an ANN.

**Dataset:** [Brain Tumor Dataset](https://www.kaggle.com/datasets/ishans24/brain-tumor-dataset)  
Download and extract so that the folder structure is:
```
brain_tumor_dataset/
    yes/   ← tumor images
    no/    ← no tumor images
```

In [ ]:
# ── Task 2 Imports ──────────────────────────────────────────────────────────
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve, auc)

# Install opencv if not available
# !pip install opencv-python -q

### Step 1 & 2: Load and Preprocess Images (Resize + Normalize)

In [ ]:
IMG_SIZE   = 64   # resize to 64×64
DATA_DIR   = 'brain_tumor_dataset'  # update if needed

def load_images(data_dir, img_size=64):
    images, labels = [], []
    class_map = {'yes': 1, 'no': 0}
    for label_name, label_val in class_map.items():
        folder = os.path.join(data_dir, label_name)
        for fname in os.listdir(folder):
            path = os.path.join(folder, fname)
            img  = cv2.imread(path)
            if img is None:
                continue
            img = cv2.resize(img, (img_size, img_size))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(label_val)
    return np.array(images), np.array(labels)

X_img, y_img = load_images(DATA_DIR, IMG_SIZE)

# Normalize pixel values to [0, 1]
X_img = X_img.astype('float32') / 255.0

print(f"Dataset shape: {X_img.shape}")
print(f"Tumor: {y_img.sum()}, No Tumor: {(y_img==0).sum()}")

# Preview a few images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    idx = np.random.randint(len(X_img))
    ax.imshow(X_img[idx])
    ax.set_title('Tumor' if y_img[idx]==1 else 'No Tumor', fontsize=8)
    ax.axis('off')
plt.suptitle('Task 2 — Sample Brain MRI Images')
plt.tight_layout()
plt.show()

In [ ]:
# Flatten images for ANN input
X_flat = X_img.reshape(len(X_img), -1)  # shape: (N, 64*64*3)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_flat, y_img, test_size=0.2, random_state=42, stratify=y_img
)
print(f"Train: {X_train2.shape}, Test: {X_test2.shape}")

### Step 3 & 4: Build & Compile ANN

In [ ]:
tf.random.set_seed(42)

model_t2 = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train2.shape[1],)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64,  activation='relu'),
    tf.keras.layers.Dense(1,   activation='sigmoid')
], name='BrainTumorANN')

model_t2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Recall(name='recall')]
)
model_t2.summary()

### Step 5: Train & Evaluate

In [ ]:
history_t2 = model_t2.fit(
    X_train2, y_train2,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Keras evaluation
results2 = model_t2.evaluate(X_test2, y_test2, verbose=0)
print(f"Test Loss:     {results2[0]:.4f}")
print(f"Test Accuracy: {results2[1]:.4f}")
print(f"Test AUC:      {results2[2]:.4f}")
print(f"Test Recall:   {results2[3]:.4f}")

# Predictions
y_pred_prob2 = model_t2.predict(X_test2, verbose=0).ravel()
y_pred2      = (y_pred_prob2 > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test2, y_pred2, target_names=['No Tumor', 'Tumor']))

# Confusion matrix
cm2 = confusion_matrix(y_test2, y_pred2)
ConfusionMatrixDisplay(cm2, display_labels=['No Tumor', 'Tumor']).plot(cmap='Greens')
plt.title('Task 2 — Confusion Matrix: Brain Tumor')
plt.tight_layout(); plt.show()

In [ ]:
# ROC Curve
fpr2, tpr2, _ = roc_curve(y_test2, y_pred_prob2)
auc_score2    = auc(fpr2, tpr2)

plt.figure(figsize=(7, 5))
plt.plot(fpr2, tpr2, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc_score2:.4f})')
plt.plot([0,1], [0,1], 'k--', lw=1)
plt.xlim([0,1]); plt.ylim([0,1.02])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Task 2 — ROC Curve: Brain Tumor Detection')
plt.legend(loc='lower right')
plt.tight_layout(); plt.show()
print(f"AUC Score: {auc_score2:.4f}")

---
# Task 3: Handwritten Digit Classification — MNIST (Multi-Class)
**Objective:** Classify handwritten digits (0–9) using an ANN.

**Dataset:** MNIST — loaded directly from TensorFlow/Keras datasets.

In [ ]:
# ── Task 3 Imports ──────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

### Step 1 & 2: Load MNIST and Normalize Pixel Values

In [ ]:
# Load MNIST directly from Keras
(X_train3, y_train3), (X_test3, y_test3) = tf.keras.datasets.mnist.load_data()

print(f"Train shape: {X_train3.shape}, Test shape: {X_test3.shape}")

# Normalize pixel values [0, 255] → [0, 1]
X_train3 = X_train3.astype('float32') / 255.0
X_test3  = X_test3.astype('float32')  / 255.0

# Visualize sample images
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
for i in range(20):
    axes[i//10, i%10].imshow(X_train3[i], cmap='gray')
    axes[i//10, i%10].set_title(str(y_train3[i]))
    axes[i//10, i%10].axis('off')
plt.suptitle('Task 3 — Sample MNIST Digits')
plt.tight_layout(); plt.show()

### Step 3: Build ANN — Flatten + Hidden Layers (ReLU) + Softmax Output (10 Classes)

In [ ]:
tf.random.set_seed(42)

model_t3 = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),   # 784 inputs
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10,  activation='softmax') # 10 classes
], name='MNIST_ANN')

model_t3.summary()

### Step 4: Compile — Adam + Categorical Cross-Entropy

In [ ]:
model_t3.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # labels are integers, not one-hot
    metrics=['accuracy']
)

### Step 5: Train and Evaluate Accuracy

In [ ]:
history_t3 = model_t3.fit(
    X_train3, y_train3,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
test_loss3, test_acc3 = model_t3.evaluate(X_test3, y_test3, verbose=0)
print(f"Test Loss:     {test_loss3:.4f}")
print(f"Test Accuracy: {test_acc3:.4f}")

# Detailed report
y_pred3 = np.argmax(model_t3.predict(X_test3, verbose=0), axis=1)
print("\nClassification Report:")
print(classification_report(y_test3, y_pred3, target_names=[str(i) for i in range(10)]))

# Confusion matrix
cm3 = confusion_matrix(y_test3, y_pred3)
plt.figure(figsize=(10, 8))
ConfusionMatrixDisplay(cm3, display_labels=list(range(10))).plot(cmap='Purples', ax=plt.gca())
plt.title('Task 3 — Confusion Matrix: MNIST Digits')
plt.tight_layout(); plt.show()

### Step 6: Plot Training vs Validation Accuracy

In [ ]:
epochs_range = range(1, len(history_t3.history['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history_t3.history['loss'],     'b-o', label='Train')
axes[0].plot(epochs_range, history_t3.history['val_loss'], 'r-o', label='Validation')
axes[0].set_title('Task 3 — Training vs Validation Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs_range, history_t3.history['accuracy'],     'b-o', label='Train')
axes[1].plot(epochs_range, history_t3.history['val_accuracy'], 'r-o', label='Validation')
axes[1].set_title('Task 3 — Training vs Validation Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# Visualize some wrong predictions
wrong_idx = np.where(y_pred3 != y_test3)[0][:10]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, idx in enumerate(wrong_idx):
    axes[i//5, i%5].imshow(X_test3[idx], cmap='gray')
    axes[i//5, i%5].set_title(f'True:{y_test3[idx]} Pred:{y_pred3[idx]}', fontsize=9, color='red')
    axes[i//5, i%5].axis('off')
plt.suptitle('Task 3 — Misclassified Digits')
plt.tight_layout(); plt.show()

---
# Task 4: Flower Image Classification (Multi-Class MLP with Backpropagation)
**Objective:** Classify flowers (5 classes) using an MLP trained with SGD (lr=0.01).

**Dataset:** [Flowers Recognition](https://www.kaggle.com/alxmamaev/flowers-recognition)  
Download and extract so the folder structure is:
```
flowers/
    daisy/
    dandelion/
    rose/
    sunflower/
    tulip/
```

In [ ]:
# ── Task 4 Imports ──────────────────────────────────────────────────────────
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve, auc)
from sklearn.preprocessing import label_binarize

### Step 1: Load and Preprocess the Flower Dataset

In [ ]:
IMG_SIZE4  = 64
DATA_DIR4  = 'flowers'  # update path if needed
CLASS_NAMES = sorted(os.listdir(DATA_DIR4))  # ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']
print("Classes:", CLASS_NAMES)

images4, labels4 = [], []

for class_name in CLASS_NAMES:
    folder = os.path.join(DATA_DIR4, class_name)
    if not os.path.isdir(folder):
        continue
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        img  = cv2.imread(path)
        if img is None:
            continue
        img = cv2.resize(img, (IMG_SIZE4, IMG_SIZE4))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images4.append(img)
        labels4.append(class_name)

X4 = np.array(images4, dtype='float32') / 255.0   # Scale to [0, 1]
print(f"Dataset shape: {X4.shape}, Total samples: {len(labels4)}")

# Encode string labels to integers
le4 = LabelEncoder()
y4  = le4.fit_transform(labels4)
print("Label mapping:", dict(zip(le4.classes_, le4.transform(le4.classes_))))

# Preview
fig, axes = plt.subplots(1, 5, figsize=(15, 4))
for i, cls in enumerate(CLASS_NAMES):
    idx = np.where(np.array(labels4) == cls)[0][0]
    axes[i].imshow(X4[idx])
    axes[i].set_title(cls.capitalize())
    axes[i].axis('off')
plt.suptitle('Task 4 — Sample Flower Images')
plt.tight_layout(); plt.show()

In [ ]:
# Flatten images for MLP
X4_flat = X4.reshape(len(X4), -1)  # (N, 64*64*3)

# One-hot encode labels for categorical_crossentropy
y4_onehot = tf.keras.utils.to_categorical(y4, num_classes=5)

# Train / Val / Test split (70% / 15% / 15%)
X_tv, X_test4,  y_tv, y_test4  = train_test_split(X4_flat, y4_onehot, test_size=0.15, random_state=42, stratify=y4)
X_train4, X_val4, y_train4, y_val4 = train_test_split(X_tv, y_tv, test_size=0.15/0.85, random_state=42)

print(f"Train: {X_train4.shape}, Val: {X_val4.shape}, Test: {X_test4.shape}")

### Steps 2 & 3: Initialize Weights and Build MLP (Forward Pass)

In [ ]:
tf.random.set_seed(42)

# MLP with 2+ hidden layers as required
# Weights initialized randomly by Keras (Glorot uniform by default)
model_t4 = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train4.shape[1],)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(5,   activation='softmax')  # 5 flower classes
], name='FlowerMLP')

model_t4.summary()

### Steps 4 & 5: Compile with SGD (lr=0.01) + Cross-Entropy Loss

In [ ]:
# SGD optimizer with learning rate = 0.01 as specified
sgd_optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)

model_t4.compile(
    optimizer=sgd_optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### Step 6: Train for 10 Epochs — Monitor Validation Accuracy

In [ ]:
history_t4 = model_t4.fit(
    X_train4, y_train4,
    epochs=10,
    batch_size=32,
    validation_data=(X_val4, y_val4),
    verbose=1
)

# Epoch-wise validation accuracy
print("\nValidation Accuracy per Epoch:")
for epoch, val_acc in enumerate(history_t4.history['val_accuracy'], 1):
    print(f"  Epoch {epoch:2d}: {val_acc:.4f}")

### Step 7: Evaluate on Test Set — Classification Report

In [ ]:
test_loss4, test_acc4 = model_t4.evaluate(X_test4, y_test4, verbose=0)
print(f"Final Test Loss:     {test_loss4:.4f}")
print(f"Final Test Accuracy: {test_acc4:.4f}")

y_pred_prob4 = model_t4.predict(X_test4, verbose=0)
y_pred4      = np.argmax(y_pred_prob4, axis=1)
y_true4      = np.argmax(y_test4, axis=1)

print("\nClassification Report:")
print(classification_report(y_true4, y_pred4, target_names=CLASS_NAMES))

# Confusion matrix
cm4 = confusion_matrix(y_true4, y_pred4)
plt.figure(figsize=(8, 6))
ConfusionMatrixDisplay(cm4, display_labels=CLASS_NAMES).plot(cmap='Oranges', ax=plt.gca())
plt.title('Task 4 — Confusion Matrix: Flower Classification')
plt.tight_layout(); plt.show()

### Step 8: Plot ROC Curve and Calculate AUC (One-vs-Rest)

In [ ]:
# Multi-class ROC using One-vs-Rest strategy
colors = ['blue', 'green', 'red', 'purple', 'orange']

plt.figure(figsize=(9, 6))

for i, (class_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr_i, tpr_i, _ = roc_curve(y_test4[:, i], y_pred_prob4[:, i])
    auc_i           = auc(fpr_i, tpr_i)
    plt.plot(fpr_i, tpr_i, color=color, lw=2,
             label=f'{class_name.capitalize()} (AUC = {auc_i:.3f})')

plt.plot([0,1], [0,1], 'k--', lw=1)
plt.xlim([0,1]); plt.ylim([0,1.02])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Task 4 — ROC Curves: Flower Classification (One-vs-Rest)')
plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
# Training curves
epochs_range4 = range(1, 11)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range4, history_t4.history['loss'],     'b-o', label='Train')
axes[0].plot(epochs_range4, history_t4.history['val_loss'], 'r-o', label='Validation')
axes[0].set_title('Task 4 — Loss per Epoch')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs_range4, history_t4.history['accuracy'],     'b-o', label='Train')
axes[1].plot(epochs_range4, history_t4.history['val_accuracy'], 'r-o', label='Validation')
axes[1].set_title('Task 4 — Accuracy per Epoch')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout(); plt.show()

---
## Summary

| Task | Problem | Architecture | Loss | Optimizer | Key Metric |
|------|---------|-------------|------|-----------|------------|
| 1 | Customer Churn | 2 Hidden (ReLU) + Sigmoid | Binary CE | Adam | F1-Score |
| 2 | Brain Tumor | 3 Hidden (ReLU) + Sigmoid | Binary CE | Adam | AUC, ROC |
| 3 | MNIST Digits | Flatten + 2 Hidden (ReLU) + Softmax | Sparse Cat CE | Adam | Accuracy |
| 4 | Flowers | 3 Hidden (ReLU) + Softmax | Categorical CE | SGD (lr=0.01) | ROC/AUC |

---
*AL-2002 Artificial Intelligence Lab — FAST-NU Lahore, Spring 2026*